# 知乎ETF轮动复现
原文连接：https://zhuanlan.zhihu.com/p/646860740

# 数据获取

In [ ]:
import os
import tushare as ts
import pandas as pd

# 设置Token（替换为你的实际Token）
ts.set_token(os.environ["TUSHARE_TOKEN"])
pro = ts.pro_api()

# ETF列表（Tushare代码）
pool_list = ['510880.SH', '159915.SZ', '510300.SH', '510500.SH']
# ETF中文名，顺序与pool_list一致
name_list = ['红利ETF', '创业板ETF', '沪深300ETF', '中证500ETF']

start_date = '20130729'
end_date = '20240729'  # 十年后

# 获取数据
data = pd.DataFrame()
for code, name in zip(pool_list, name_list):
    df = pro.fund_daily(
        ts_code=code,
        start_date=start_date,
        end_date=end_date,
        fields='trade_date,close'
    )
    df.rename(columns={'trade_date': 'date', 'close': name}, inplace=True)
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
    if data.empty:
        data = df
    else:
        data = pd.merge(data, df, on='date', how='outer')

# 按顺序排列列
data = data[['date'] + name_list]

# 按日期升序排列
data = data.sort_values('date').reset_index(drop=True)

# 输出最早的10天数据
print("Tushare获取的ETF历史数据（最早10天）")
print("="*60)
print(data[['date'] + name_list].head(10))
print('name_list:', name_list)

# 保存为Excel（可选）
data.to_excel('etf_data_tushare.xlsx', index=False, float_format='%.3f')

# 数据计算

In [ ]:
# 动量长度
N = 20
# 计算每日涨跌幅和N日涨跌幅
for name in name_list:
    data['日收益率_'+name] = data[name] / data[name].shift(1) - 1.0
    data['涨幅_'+name] = data[name] / data[name].shift(N+1) - 1.0
# 去掉缺失值
data = data.dropna()
data[['date'] + ['涨幅_'+v for v in name_list]].head(10)

# 信号生成与回测

In [ ]:
# 取出每日涨幅最大的证券
data['信号'] = data[['涨幅_'+v for v in name_list]].idxmax(axis=1).str.replace('涨幅_', '')
# 今日的涨幅由昨日的持仓产生
data['信号'] = data['信号'].shift(1)
data = data.dropna()
data['轮动策略日收益率'] = data.apply(lambda x: x['日收益率_'+x['信号']], axis=1) 
# 第一天尾盘交易，当日涨幅不纳入
data.loc[data.index[0],'轮动策略日收益率'] = 0.0
data['轮动策略净值'] = (1.0 + data['轮动策略日收益率']).cumprod()
data[['date'] + ['涨幅_'+v for v in name_list]+['信号','轮动策略日收益率','轮动策略净值']].head(10)

# 结果统计与画图

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 显示中文设置
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False

# 确保date为datetime类型
data['date'] = pd.to_datetime(data['date'])

# 绘制净值曲线图
fig, ax = plt.subplots(figsize=(15, 6))
ax.set_xlabel('日期')
ax.set_ylabel('净值')

for name in name_list+['轮动策略']:
    if name in name_list:
        data[name+'净值'] = data[name]/data[name].iloc[0]
        ax.plot(data['date'], data[name+'净值'].values, linestyle='--')
    else:
        ax.plot(data['date'], data[name+'净值'].values, linestyle='-', color='#FF8124')

# 设置横坐标为年份
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 显示图例和标题
ax.legend(name_list+['轮动策略'])
ax.set_title('轮动策略净值曲线对比 by 公众号【量化君也】')

plt.show()

# 回测报告

In [ ]:
import quantstats as qstat

# 如果date还在columns里，才做转换和set_index
if 'date' in data.columns:
    data['date'] = pd.to_datetime(data['date'])
    data = data.set_index('date')

#将完整回测报告存为HTML文件
qstat.reports.html(data['轮动策略净值'], benchmark=data['沪深300ETF净值'],
                  title='轮动策略回测报告 by 公众号【量化君也】',
                  download_filename='轮动策略回测报告 by 公众号【量化君也】.html') 
#输出基本回测报告信息
qstat.reports.basic(data['轮动策略净值'], benchmark=data['沪深300ETF净值'])

# 轮动策略改进1
可以先从候选池下手。

侯选池的重要性不言而喻，因为后续所有的动作都是围绕着侯选池展开的，就跟从中国足球队随机挑选球员，还是从巴西足球队随机挑选球员去点球一样，底子好的自然胜率高。

比如说，咱除了A股之外，还考虑国外市场，选入代表漂亮国的纳指ETF（代码：513100），还考虑大宗商品市场，于是选入黄金ETF（代码：518880），同时剔除原来候选池当中的沪深300ETF和中证500ETF，于是整个候选池就变为：红利ETF（510880），创业板ETF（159915），纳指ETF（513100），黄金ETF（518880）。于是乎，获取数据部分的代码做出对应的调整。

获取数据太恶心了，不做了

# 轮动策略改进2

候选池改过了，咱接下来就是改强弱排序，还是保存动量规则不变，原来采用的是区间涨跌幅，这次改进则打算采用区间走势

区间走势用什么指标来衡量呢？这里借鉴RSRS指标的构建思路，用收盘价序列的斜率来表征，斜率越大，走势越猛越强。

同时也引入决定系数R2的概念，它是对线性拟合效果好坏的判断指标，取值范围一般在0~1之间，数值越大，表示线性拟合的效果就越好，当直线能完美拟合所有数据点时，取值为1，更详细的说明可以见之前的文章

https://mp.weixin.qq.com/s?__biz=MzkyODI5ODcyMA==&mid=2247485225&idx=1&sn=dba12368183359f98f7b0f60ea6a38a4&chksm=c21ba4a6f56c2db0684151422eb24311f06902d656ce3654bfd4954aceeb4571cdbdd26712dd&scene=21#wechat_redirect。

于是强弱规则就修改为“斜率和决定系数的乘积”，乘积作为ETF动量强弱的得分，得分数值越高，就表示动量越强，每日都选入强弱得分最高的ETF，数据计算部分的代码也做出相应的调整。

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# 计算强弱得分
def calculate_score(srs, N=25):
    if srs.shape[0] < N:
        return np.nan
    x = np.arange(1, N+1)
    y = srs.values / srs.values[0]
    lr = LinearRegression().fit(x.reshape(-1, 1), y)
    # 斜率
    slope = lr.coef_[0]
    # 决定系数R2
    r_squared = lr.score(x.reshape(-1, 1), y)
    # 得分
    score = 10000 * slope * r_squared
    return score

# 斜率计算长度
N = 25
# 计算每日涨跌幅和得分
for name in name_list:
    data['日收益率_'+name] = data[name] / data[name].shift(1) - 1.0
    data['得分_'+name] = data[name].rolling(N).apply(lambda x: calculate_score(x, N))
# 去掉缺失值
data = data.dropna()
data[['得分_'+v for v in name_list]].head(10)